In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:

class LinearNoiseScheduler:
    def __init__(self, num_timesteps, beta_start, beta_end):
        self.num_timesteps = num_timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps)
        self.alphas = 1.0 - self.betas
        self.alpha_cum_prod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1.0 - self.alpha_cum_prod)

    def add_noise(self, original, noise, t):
        original_shape = original.shape
        batch_size = original_shape[0]
        sqrt_alpha_cum_prod = self.sqrt_alpha_cum_prod.to(original.device)[t].reshape(batch_size)
        sqrt_one_minus_alpha_cum_prod = self.sqrt_one_minus_alpha_cum_prod.to(original.device)[t].reshape(batch_size)
        for i in range(len(original_shape) - 1):
            sqrt_alpha_cum_prod = sqrt_alpha_cum_prod.unsqueeze(-1)
            sqrt_one_minus_alpha_cum_prod = sqrt_one_minus_alpha_cum_prod.unsqueeze(-1)
        return sqrt_alpha_cum_prod * original + sqrt_one_minus_alpha_cum_prod * noise

    def sample_prev_timestep(self, xt, noise_pred, t):
        device = xt.device
        beta_t = self.betas[t].to(device)
        alpha_t = self.alphas[t].to(device)
        sqrt_alpha_cum_prod_t = self.sqrt_alpha_cum_prod[t].to(device)
        sqrt_one_minus_alpha_cum_prod_t = self.sqrt_one_minus_alpha_cum_prod[t].to(device)
        alpha_cum_prod_t = self.alpha_cum_prod[t].to(device)

        x0 = (xt - (sqrt_one_minus_alpha_cum_prod_t * noise_pred)) / sqrt_alpha_cum_prod_t
        x0 = torch.clamp(x0, -1., 1.)

        mean = xt - ((beta_t * noise_pred) / sqrt_one_minus_alpha_cum_prod_t)
        mean = mean / torch.sqrt(alpha_t)

        if t == 0:
            return mean, x0
        else:
            alpha_cum_prod_t_prev = self.alpha_cum_prod[t-1].to(device)
            variance = (1 - alpha_cum_prod_t_prev) / (1 - alpha_cum_prod_t)
            variance = variance * beta_t
            sigma = variance ** 0.5
            z = torch.randn(xt.shape).to(device)
            return mean + sigma * z, x0

In [ ]:
def get_time_embedding(time_steps, t_emb_dim):
    factor = 10000 ** ((torch.arange(0, t_emb_dim // 2, device=time_steps.device) / (t_emb_dim // 2)))
    t_emb = time_steps[:, None].repeat(1, t_emb_dim // 2) / factor
    t_emb = torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim=-1)
    return t_emb

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, down_sample, num_heads):
        super().__init__()
        self.down_sample = down_sample
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels)
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        )
        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.down_sample_conv = nn.Conv2d(out_channels, out_channels, kernel_size=4, stride=2, padding=1) if self.down_sample else nn.Identity()

    def forward(self, x, t_emb):
        out = x
        resnet_input = out
        out = self.resnet_conv_first(out)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(resnet_input)
        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn)
        in_attn = in_attn.transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn
        skip = out
        out = self.down_sample_conv(out)
        return out, skip

In [ ]:
class MidBlock(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels, t_emb_dim, num_heads):
        super().__init__()
        self.resnet_conv_first = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, in_channels),
                nn.SiLU(),
                nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1)
            ),
            nn.Sequential(
                nn.GroupNorm(8, mid_channels),
                nn.SiLU(),
                nn.Conv2d(mid_channels, out_channels, kernel_size=3, stride=1, padding=1)
            )
        ])
        self.resnet_conv_second = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, mid_channels),
                nn.SiLU(),
                nn.Conv2d(mid_channels, mid_channels, kernel_size=3, stride=1, padding=1)
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
            )
        ])
        self.t_emb_layers = nn.ModuleList([
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, mid_channels)),
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels))
        ])
        self.attention_norm = nn.GroupNorm(8, mid_channels)
        self.attention = nn.MultiheadAttention(mid_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.ModuleList([
            nn.Conv2d(in_channels, mid_channels, kernel_size=1),
            nn.Conv2d(mid_channels, out_channels, kernel_size=1)
        ])

    def forward(self, x, t_emb):
        out = x
        resnet_input = out
        out = self.resnet_conv_first[0](out)
        out = out + self.t_emb_layers[0](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[0](out)
        out = out + self.residual_input_conv[0](resnet_input)
        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn)
        in_attn = in_attn.transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn
        resnet_input = out
        out = self.resnet_conv_first[1](out)
        out = out + self.t_emb_layers[1](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[1](out)
        out = out + self.residual_input_conv[1](resnet_input)
        return out

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, up_sample, num_heads):
        super().__init__()
        self.up_sample = up_sample
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels)
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        )
        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.up_sample_conv = nn.ConvTranspose2d(out_channels, out_channels, kernel_size=4, stride=2, padding=1) if self.up_sample else nn.Identity()

    def forward(self, x, out_down, t_emb):
        x = torch.cat([x, out_down], dim=1)
        out = x
        resnet_input = out
        out = self.resnet_conv_first(out)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(resnet_input)
        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn)
        in_attn = in_attn.transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn
        out = self.up_sample_conv(out)
        return out

In [ ]:
class Unet(nn.Module):
    def __init__(self, im_channels):
        super().__init__()
        self.down_channels = [32, 64, 128, 256]
        self.mid_channels = [256, 256, 128]
        self.t_emb_dim = 128
        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim)
        )
        self.init_conv = nn.Conv2d(im_channels, self.down_channels[0], kernel_size=3, padding=1)

        self.down_blocks = nn.ModuleList()
        self.skip_channels = []
        for i in range(len(self.down_channels) - 1):
            in_channels = self.down_channels[i]
            out_channels = self.down_channels[i + 1]
            down_sample = (i < len(self.down_channels) - 2)
            self.down_blocks.append(DownBlock(in_channels, out_channels, self.t_emb_dim, down_sample, num_heads=4))
            self.skip_channels.append(out_channels)

        self.mid_block = MidBlock(self.mid_channels[0], self.mid_channels[1], self.mid_channels[2], self.t_emb_dim, num_heads=4)

        self.up_blocks = nn.ModuleList()
        current_channels = self.mid_channels[-1]
        for i in reversed(range(len(self.down_channels) - 1)):
            skip_c = self.skip_channels[i]
            in_channels = current_channels + skip_c
            out_channels = self.down_channels[i]
            up_sample = (i > 0)
            self.up_blocks.append(UpBlock(in_channels, out_channels, self.t_emb_dim, up_sample, num_heads=4))
            current_channels = out_channels

        self.final_conv = nn.Sequential(
            nn.GroupNorm(8, self.down_channels[0]),
            nn.SiLU(),
            nn.Conv2d(self.down_channels[0], im_channels, kernel_size=3, padding=1)
        )

    def forward(self, x, t):
        t_emb = get_time_embedding(t, self.t_emb_dim)
        t_emb = self.t_proj(t_emb)
        out = self.init_conv(x)
        down_outputs = []
        for block in self.down_blocks:
            out, skip = block(out, t_emb)
            down_outputs.append(skip)
        out = self.mid_block(out, t_emb)
        for block in self.up_blocks:
            down_out = down_outputs.pop()
            out = block(out, down_out, t_emb)
        out = self.final_conv(out)
        return out

In [ ]:
def print_model_config(model, T):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n" + "="*50)
    print("MODEL CONFIG")
    print("="*50)
    print(f"Total Timesteps (T)  : {T}")
    print(f"Total Parameters     : {total_params:,}")
    print(f"Trainable Parameters : {trainable_params:,}")
    print(f"Time Embedding Dim   : {model.t_emb_dim}")
    print(f"Down Channels Path   : {model.down_channels}")
    print(f"Mid Channels Path    : {model.mid_channels}")
    print("="*50 + "\n")

In [ ]:
def train_model(model, scheduler, dataloader, optimizer, criterion, epochs, T, device):
    model.train()
    epoch_losses = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch_idx, (x, _) in enumerate(pbar):
            x = x.to(device)
            batch_size = x.shape[0]

            t = torch.randint(0, T, (batch_size,), device=device)
            noise = torch.randn_like(x)

            noisy_x = scheduler.add_noise(x, noise, t)

            noise_pred = model(noisy_x, t)

            loss = criterion(noise_pred, noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = epoch_loss / len(dataloader)
        epoch_losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}")

    return epoch_losses

In [ ]:
def plot_losses(losses, save_path="loss_plot.png"):
    """Plots the training loss curve and saves it to a file."""
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(losses) + 1), losses, marker='o', linestyle='-', color='b')
    plt.title('Diffusion Model Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Average MSE Loss')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(save_path)
    print(f"\nLoss plot saved to '{save_path}'")
    plt.show()

In [ ]:

def generate_and_save_gif(model, scheduler, T, device, save_path="mnist_generation.gif"):
    model.eval()
    frames = []
    print("\nGenerating Inference GIF...")

    with torch.no_grad():
        x = torch.randn(1, 1, 32, 32).to(device)

        for i in tqdm(reversed(range(T)), desc="Sampling", total=T):
            t_tensor = torch.tensor([i], device=device)
            noise_pred = model(x, t_tensor)
            x, _ = scheduler.sample_prev_timestep(x, noise_pred, i)

            if i % 10 == 0 or i == 0:
                img = x.squeeze().cpu().numpy()
                img = np.clip((img + 1.0) / 2.0, 0.0, 1.0)
                img = (img * 255).astype(np.uint8)

                img_pil = Image.fromarray(img).resize((128, 128), Image.NEAREST)
                frames.append(img_pil)

    frames[0].save(
        save_path,
        save_all=True,
        append_images=frames[1:],
        duration=40,
        loop=0
    )
    print(f"Inference complete. Saved to '{save_path}'")

In [ ]:
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size = 64
    epochs = 10
    T = 1000
    im_channels = 1

    transform = transforms.Compose([
        transforms.Pad(2),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2, drop_last=True)

    scheduler = LinearNoiseScheduler(num_timesteps=T, beta_start=1e-4, beta_end=0.02)
    model = Unet(im_channels=im_channels).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
    criterion = nn.MSELoss()

    print_model_config(model, T)

    losses = train_model(model, scheduler, dataloader, optimizer, criterion, epochs, T, device)

    plot_losses(losses, save_path="training_loss.png")

    generate_and_save_gif(model, scheduler, T, device, save_path="mnist_generation.gif")

In [ ]:
import torch
from torchvision.utils import make_grid
from PIL import Image
import numpy as np
from tqdm import tqdm

def generate_15x15_grid_gif(model, scheduler, T, device, save_path="mnist_15x15_grid.gif"):
    model.eval()
    frames = []

    num_images = 25
    grid_nrow = 5

    print(f"\nGenerating {num_images} digits in a {grid_nrow}x{grid_nrow} grid...")

    with torch.no_grad():
        x = torch.randn(num_images, 1, 32, 32).to(device)

        for i in tqdm(reversed(range(T)), desc="Sampling Grid", total=T):
            t_tensor = torch.full((num_images,), i, device=device, dtype=torch.long)

            noise_pred = model(x, t_tensor)
            x, _ = scheduler.sample_prev_timestep(x, noise_pred, i)

            if i % 10 == 0 or i == 0:
                x_norm = torch.clamp((x + 1.0) / 2.0, min=0.0, max=1.0)

                grid = make_grid(x_norm, nrow=grid_nrow, padding=1, pad_value=1.0)

                ndarr = grid.mul(255).add_(0.5).clamp_(0, 255).permute(1, 2, 0).to('cpu', torch.uint8).numpy()
                img_pil = Image.fromarray(ndarr)

                img_pil = img_pil.resize((img_pil.width * 2, img_pil.height * 2), Image.NEAREST)
                frames.append(img_pil)

    frames[0].save(
        save_path,
        save_all=True,
        append_images=frames[1:],
        duration=40,
        loop=0
    )
    print(f"Grid inference complete. Saved to '{save_path}'")

generate_15x15_grid_gif(model, scheduler, T, device, "mnist_5x5_grid.gif")